# ForecastLLM - Week 7 Day 1: LoRA and QLoRA for Forecasting

This notebook adapts the original Week 7 Day 1 quantization/LoRA setup from product pricing to time-series forecasting.

## What changed vs. original

- Domain changed from product-price text to forecasting prompts built from `timestamp/value` series data.
- We reuse Week 6 data and feature pipeline (lags + calendar features) as prompt context.
- Quantization experiments are kept, with safe fallback messages for CPU-only environments.


## Runtime Note

8-bit and 4-bit loading via `bitsandbytes` typically requires a CUDA GPU.
If your environment is CPU-only, the quantized cells will report a skip message and continue.

## DGX Spark (ARM64) Adjustments

This notebook was adapted from Colab usage, but your target is local DGX Spark on Ubuntu ARM64.

- Authentication: use local `HF_TOKEN` env var or `huggingface-cli login` (no Colab secrets).
- Precision: Hopper/Blackwell generally benefits from `bfloat16` compute defaults.
- Quantization: `bitsandbytes` now supports Linux `aarch64`, but environment mismatches can still happen; we run capability checks before 8-bit/4-bit loads.

In [ ]:
# Optional install for DGX Spark (Ubuntu ARM64 + CUDA)
# Recommendation: keep the system/NVIDIA PyTorch build if already present.
# If you need to install Python-side deps, run this cell.

# %pip install -q --upgrade \
#   "transformers>=4.48,<5" \
#   "accelerate>=0.34,<2" \
#   "peft>=0.13,<1" \
#   "datasets>=3,<5" \
#   "trl>=0.12,<1" \
#   "bitsandbytes>=0.49.2"

# ARM64 fallback (only if the normal bitsandbytes install is incompatible):
# %pip install --force-reinstall --no-deps \
#   https://github.com/bitsandbytes-foundation/bitsandbytes/releases/download/continuous-release_main/bitsandbytes-1.33.7.preview-py3-none-manylinux_2_24_aarch64.whl


In [2]:
# imports

import os
import gc
import sys
import numpy as np
import pandas as pd
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

from week6.data_loader import load_sample_series

In [3]:
# Constants

TRAIN_FRACTION = 0.8
SEASONAL_PERIOD = 7

# Pick a smaller open model by default, so this notebook is practical locally
BASE_MODEL = os.getenv("BASE_MODEL", "TinyLlama/TinyLlama-1.1B-Chat-v1.0")
FINETUNED_MODEL = os.getenv("FINETUNED_MODEL", "")

print(f"BASE_MODEL={BASE_MODEL}")
print("CUDA available:", torch.cuda.is_available())

# Optional local auth for gated/private models
# export HF_TOKEN=... && huggingface-cli login --token "$HF_TOKEN"
HF_TOKEN = os.getenv("HF_TOKEN", "")


BASE_MODEL=TinyLlama/TinyLlama-1.1B-Chat-v1.0
CUDA available: True


In [4]:
# Environment probe (helpful on DGX Spark ARM64)
import platform

print('Platform:', platform.platform())
print('Machine:', platform.machine())
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('Torch CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    dev = torch.cuda.get_device_name(0)
    cc = torch.cuda.get_device_capability(0)
    print('GPU 0:', dev)
    print('Compute capability:', cc)

try:
    import bitsandbytes as bnb
    print('bitsandbytes:', bnb.__version__)
except (ImportError, ModuleNotFoundError, OSError) as e:
    print('bitsandbytes import failed:', e)

Platform: Linux-6.17.0-1014-nvidia-aarch64-with-glibc2.39
Machine: aarch64
Python: 3.12.12
PyTorch: 2.11.0+cu130
Torch CUDA: 13.0
CUDA available: True
GPU 0: NVIDIA GB10
Compute capability: (12, 1)
bitsandbytes: 0.49.2


In [5]:
# Version sanity check
try:
    import transformers, peft, accelerate
    print('transformers:', transformers.__version__)
    print('peft:', peft.__version__)
    print('accelerate:', accelerate.__version__)
except (ImportError, ModuleNotFoundError, OSError) as e:
    print('Core package check failed:', e)


transformers: 5.6.2
peft: 0.19.1
accelerate: 1.13.0


In [6]:
# CUDA math settings (safe defaults for Hopper/Blackwell)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
    print('Enabled TF32 + high matmul precision.')
else:
    print('CUDA not available; skipped CUDA math settings.')

Enabled TF32 + high matmul precision.


## Reuse Week 6 Forecasting Pipeline

Before model loading, we build a supervised forecasting table so later fine-tuning steps are grounded in the same data protocol used in Week 6.

In [7]:
# Load and normalize to [timestamp, value]
def make_synthetic_series(periods=240):
    dates = pd.date_range("2024-01-01", periods=periods, freq="D")
    trend = np.linspace(80, 110, periods)
    weekly = 5 * np.sin(np.arange(periods) * 2 * np.pi / 7)
    noise = np.random.default_rng(42).normal(scale=1.0, size=periods)
    return pd.DataFrame({"timestamp": dates, "value": trend + weekly + noise})

try:
    loaded = load_sample_series()
except (FileNotFoundError, ValueError, TypeError, OSError) as e:
    print(f"Loader failed ({e}); using fallback")
    loaded = None

if isinstance(loaded, pd.Series):
    ts_df = loaded.rename("value").to_frame().reset_index()
    if ts_df.shape[1] == 2:
        ts_df.columns = ["timestamp", "value"]
elif isinstance(loaded, pd.DataFrame):
    ts_df = loaded.copy()
else:
    ts_df = make_synthetic_series()

if "value" not in ts_df.columns:
    numeric_cols = ts_df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        ts_df = ts_df.rename(columns={numeric_cols[0]: "value"})
    else:
        ts_df = make_synthetic_series()

if "timestamp" not in ts_df.columns:
    ts_df = ts_df.reset_index().rename(columns={ts_df.index.name or "index": "timestamp"})

ts_df = ts_df[["timestamp", "value"]].copy()
ts_df["timestamp"] = pd.to_datetime(ts_df["timestamp"], errors="coerce")
ts_df["value"] = pd.to_numeric(ts_df["value"], errors="coerce")
ts_df = ts_df.dropna(subset=["value"]).sort_values("timestamp").reset_index(drop=True)

if ts_df["timestamp"].isna().all() or len(ts_df) < 60:
    ts_df = make_synthetic_series()

print(f"Loaded {len(ts_df):,} rows")
ts_df.head()

Loaded 240 rows


,timestamp,value
0,2024-01-01,80.304717
1,2024-01-02,82.994696
2,2024-01-03,85.876137
3,2024-01-04,83.486552
4,2024-01-05,76.381638


In [8]:
# Build supervised lag/calendar features
supervised_df = ts_df.copy()
supervised_df["lag_1"] = supervised_df["value"].shift(1)
supervised_df["lag_2"] = supervised_df["value"].shift(2)
supervised_df["lag_3"] = supervised_df["value"].shift(3)
supervised_df["lag_7"] = supervised_df["value"].shift(7)
supervised_df["day_of_week"] = supervised_df["timestamp"].dt.dayofweek
supervised_df["month"] = supervised_df["timestamp"].dt.month
supervised_df = supervised_df.dropna().reset_index(drop=True)

FEATURE_COLUMNS = ["lag_1", "lag_2", "lag_3", "lag_7", "day_of_week", "month"]
split_idx = int(len(supervised_df) * TRAIN_FRACTION)
train_supervised = supervised_df.iloc[:split_idx].copy()
test_supervised = supervised_df.iloc[split_idx:].copy()

print("Train rows:", len(train_supervised), "| Test rows:", len(test_supervised))
train_supervised.head()

Train rows: 186 | Test rows: 47


,timestamp,value,lag_1,lag_2,lag_3,lag_7,day_of_week,month
0,2024-01-08,80.562418,76.971821,74.450796,76.381638,80.304717,0,1
1,2024-01-09,84.896540,80.562418,76.971821,74.450796,82.994696,1,1
2,2024-01-10,85.151303,84.896540,80.562418,76.971821,85.876137,2,1
3,2024-01-11,84.304047,85.151303,84.896540,80.562418,83.486552,3,1
4,2024-01-12,79.989126,84.304047,85.151303,84.896540,76.381638,4,1


## Forecast Prompt Template

We turn each supervised row into an instruction-style prompt that asks for the next value estimate.

In [9]:
def build_forecast_prompt(row):
    return (
        "You are a forecasting assistant.\n"
        "Given recent lags and calendar context, predict the next daily value.\n"
        f"lag_1={row['lag_1']:.3f}, lag_2={row['lag_2']:.3f}, lag_3={row['lag_3']:.3f}, lag_7={row['lag_7']:.3f}, "
        f"day_of_week={int(row['day_of_week'])}, month={int(row['month'])}.\n"
        "Return only a numeric forecast."
    )

def build_forecast_target(row):
    return f"{row['value']:.3f}"

example_row = train_supervised.iloc[0]
print("PROMPT:\n")
print(build_forecast_prompt(example_row))
print("\nTARGET:", build_forecast_target(example_row))

PROMPT:

You are a forecasting assistant.
Given recent lags and calendar context, predict the next daily value.
lag_1=76.972, lag_2=74.451, lag_3=76.382, lag_7=80.305, day_of_week=0, month=1.
Return only a numeric forecast.

TARGET: 80.562


In [10]:
train_records = [
    {
        "prompt": build_forecast_prompt(r),
        "completion": build_forecast_target(r),
    }
    for _, r in train_supervised.iterrows()
]

print("Training prompt rows prepared:", len(train_records))
pd.DataFrame(train_records).head(2)

Training prompt rows prepared: 186


,prompt,completion
0,You are a forecasting assistant.\nGiven recent...,80.562
1,You are a forecasting assistant.\nGiven recent...,84.897


## Quantization Setup

We now mirror the original Day 1 flow: compare full precision, 8-bit, and 4-bit loading footprint.

In [11]:
def clear_model(model=None):
    if model is not None:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def memory_gb(model):
    try:
        return model.get_memory_footprint() / 1e9
    except (AttributeError, TypeError, RuntimeError):
        return float("nan")

def load_tokenizer(model_name: str):
    from_pretrained = getattr(AutoTokenizer, "from_pretrained", None)
    if from_pretrained is None:
        raise RuntimeError("AutoTokenizer.from_pretrained is unavailable")

    loaded_tokenizer = from_pretrained(model_name)
    if loaded_tokenizer is None:
        raise RuntimeError(f"Tokenizer load failed for {model_name}")

    if getattr(loaded_tokenizer, "pad_token", None) is None:
        eos_token = getattr(loaded_tokenizer, "eos_token", None)
        unk_token = getattr(loaded_tokenizer, "unk_token", None)
        if eos_token is not None:
            loaded_tokenizer.pad_token = eos_token
        elif unk_token is not None:
            loaded_tokenizer.pad_token = unk_token
    return loaded_tokenizer

In [12]:
# Load base model without quantization
tokenizer = load_tokenizer(BASE_MODEL)

base_model_fp = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

print(f"Full precision footprint: {memory_gb(base_model_fp):,.2f} GB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:12<00:00, 15.96it/s]


Full precision footprint: 2.20 GB


In [13]:
# quick inference smoke test
messages = build_forecast_prompt(test_supervised.iloc[0])
tokenizer_call = getattr(tokenizer, "__call__", None)
if tokenizer_call is None:
    raise RuntimeError("Tokenizer is not callable")
inputs = tokenizer_call(messages, return_tensors="pt")
inputs = {k: v.to(base_model_fp.device) for k, v in inputs.items()}

with torch.no_grad():
    out = base_model_fp.generate(**inputs, max_new_tokens=24, do_sample=False)

decode_fn = getattr(tokenizer, "decode", None)
if decode_fn is None:
    raise RuntimeError("Tokenizer decode is unavailable")
decoded = decode_fn(out[0], skip_special_tokens=True)
print(decoded[-400:])

[transformers] Both `max_new_tokens` (=24) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


You are a forecasting assistant.
Given recent lags and calendar context, predict the next daily value.
lag_1=105.887, lag_2=108.670, lag_3=109.482, lag_7=101.361, day_of_week=4, month=7.
Return only a numeric forecast.
Return a list of tuples containing the predicted daily value and the corresponding lag.
Return a list of tuples


## 8-bit Trial

If this cell fails due to environment constraints, continue to the next section.

In [14]:
base_model_8bit = None

if torch.cuda.is_available():
    try:
        quant_8 = BitsAndBytesConfig(load_in_8bit=True)
        base_model_8bit = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=quant_8,
            device_map="auto",
        )
        print(f"8-bit footprint: {memory_gb(base_model_8bit):,.2f} GB")
    except (RuntimeError, ValueError, OSError, ImportError) as e:
        print("8-bit load failed:", e)
else:
    print("Skipping 8-bit load: CUDA not available.")

Loading weights: 100%|██████████| 201/201 [00:12<00:00, 16.40it/s]


8-bit footprint: 1.23 GB


## 4-bit Trial (QLoRA-style base loading)

This is the setup typically used for parameter-efficient fine-tuning.

In [15]:
base_model_4bit = None

if torch.cuda.is_available():
    try:
        quant_4 = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
        )
        base_model_4bit = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=quant_4,
            device_map="auto",
        )
        sys.stdout.write(f"4-bit footprint: {memory_gb(base_model_4bit):,.2f} GB\n")
    except (RuntimeError, ValueError, OSError, ImportError) as e:
        sys.stdout.write(f"4-bit load failed: {e}\n")
else:
    sys.stdout.write("Skipping 4-bit load: CUDA not available.\n")

Loading weights:   0%|          | 1/201 [00:00<02:23,  1.39it/s]/home/geo/Projects/Python/forecastllm/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 201/201 [00:11<00:00, 17.56it/s]


4-bit footprint: 0.75 GB


In [16]:
# Optional: attach an existing LoRA adapter if FINETUNED_MODEL is set
fine_tuned_model = None

if FINETUNED_MODEL and base_model_4bit is not None:
    try:
        fine_tuned_model = PeftModel.from_pretrained(base_model_4bit, FINETUNED_MODEL)
        print(f"4-bit + LoRA footprint: {memory_gb(fine_tuned_model):,.2f} GB")
    except (RuntimeError, ValueError, OSError, ImportError) as e:
        print("Could not load FINETUNED_MODEL:", e)
else:
    print("Set FINETUNED_MODEL env var to load an adapter checkpoint.")

Set FINETUNED_MODEL env var to load an adapter checkpoint.


## Understanding LoRA Weights and Dimensions

We keep the same Day 1 intuition: LoRA adds low-rank trainable matrices to selected modules instead of updating all base weights.

In [17]:
def lora_size_estimate(hidden_size, intermediate_size, num_layers, r, include_mlp=False):
    # Attention projections: q, k, v, o
    attn = (hidden_size * r + hidden_size * r) * 4

    mlp = 0
    if include_mlp:
        # gate, up, down projections (approximation)
        mlp = (hidden_size * r + intermediate_size * r) * 3

    params = (attn + mlp) * num_layers
    size_mb = (params * 4) / 1_000_000
    return params, size_mb

lite_params, lite_mb = lora_size_estimate(
    hidden_size=2048,
    intermediate_size=5632,
    num_layers=22,
    r=32,
    include_mlp=False,
)

full_params, full_mb = lora_size_estimate(
    hidden_size=2048,
    intermediate_size=5632,
    num_layers=22,
    r=128,
    include_mlp=True,
)

print(f"Lite LoRA (r=32, attention-only): {lite_params:,} params | {lite_mb:,.1f} MB")
print(f"Larger LoRA (r=128, attention+MLP): {full_params:,} params | {full_mb:,.1f} MB")

Lite LoRA (r=32, attention-only): 11,534,336 params | 46.1 MB
Larger LoRA (r=128, attention+MLP): 111,017,984 params | 444.1 MB


## Day 1 Wrap-up

- We converted Week 6 supervised features into instruction-style forecasting prompts.
- We compared model loading approaches (full precision vs. quantized) for practical GPU usage.
- We reviewed LoRA parameter scaling to prepare for fine-tuning in upcoming days.

Day 2 will package forecasting prompts into train/validation artifacts for SFT.